# Experiment 8 - Semantic Search using Embeddings and FAISS


In [ ]:
!pip -q install sentence-transformers faiss-cpu pandas

In [ ]:
# Small educational corpus based on the public BattingCoach README/project description.
# It summarizes project claims; it is not a substitute for the repository source or a coaching manual.
documents = [
 {"doc_id":"BC01","title":"Project overview","source":"BattingCoach README",
  "text":"BatCoach AI Pro is described as a browser-based cricket batting laboratory and virtual coach. It analyzes webcam video and provides feedback about batting technique."},
 {"doc_id":"BC02","title":"Pose and biomechanics","source":"BattingCoach README",
  "text":"The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its biomechanics module evaluates cues such as lead elbow elevation, lead knee flexion, head-over-knee balance, and spine forward lean."},
 {"doc_id":"BC03","title":"Practice modes","source":"BattingCoach README",
  "text":"The README describes shadow practice without a bat and live willow practice with a bat. Shadow practice bypasses bat-specific checks. Live willow mode includes bat tracking."},
 {"doc_id":"BC04","title":"Computer vision and shot classification","source":"BattingCoach README",
  "text":"The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a VideoMAE transformer for classifying cricket shots across ten foundational shot types."},
 {"doc_id":"BC05","title":"Feedback and repeated errors","source":"BattingCoach README",
  "text":"The project describes on-screen coaching cues and voice feedback. It tracks total swings and clean repetitions, and repeated technique errors can trigger an intervention that freezes the rep counter."},
 {"doc_id":"BC06","title":"Limitations and responsible use","source":"Notebook-authored evaluation note",
  "text":"Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence and the quality of the input video. A single recording cannot guarantee future match performance or diagnose a medical condition. Treat automated feedback as an aid for practice, not a replacement for a qualified coach."},
]
chunks = []
for d in documents:
    # short chunks preserve the source title and unique chunk id
    chunks.append({"chunk_id":d["doc_id"]+"-C1","title":d["title"],"source":d["source"],"text":d["text"]})
print("Documents:",len(documents),"Chunks:",len(chunks))
for c in chunks: print(c["chunk_id"],"|",c["title"],"|",c["text"][:90]+"...")

Documents: 6 Chunks: 6
BC01-C1 | Project overview | BatCoach AI Pro is described as a browser-based cricket batting laboratory and virtual coa...
BC02-C1 | Pose and biomechanics | The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its ...
BC03-C1 | Practice modes | The README describes shadow practice without a bat and live willow practice with a bat. Sh...
BC04-C1 | Computer vision and shot classification | The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a Video...
BC05-C1 | Feedback and repeated errors | The project describes on-screen coaching cues and voice feedback. It tracks total swings a...
BC06-C1 | Limitations and responsible use | Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence an...


In [ ]:
import numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)
texts=[c["text"] for c in chunks]
embeddings=embedder.encode(texts,convert_to_numpy=True,normalize_embeddings=True).astype("float32")
print("Embedding model:",EMBED_MODEL)
print("Embedding shape:",embeddings.shape)
index=faiss.IndexFlatIP(embeddings.shape[1]) # inner product = cosine similarity after normalization
index.add(embeddings)
print("FAISS index size:",index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (6, 384)
FAISS index size: 6


In [ ]:
queries = [
 "What technologies estimate body pose and joint angles?",
 "How does the system track the bat when a player uses one?",
 "What happens when the same technique mistake is repeated?",
 "Can the webcam guarantee a century in the next match?",
 "What is the difference between shadow practice and live willow mode?"
]
TOP_K=3
retrieval_rows=[]
for query in queries:
    qv=embedder.encode([query],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores, ids=index.search(qv,TOP_K)
    print("\nQUERY:",query)
    for rank,(score,idx) in enumerate(zip(scores[0],ids[0]),start=1):
        c=chunks[int(idx)]
        print(f"Rank {rank} | {c['chunk_id']} | cosine={score:.3f} | {c['title']}\n{c['text']}")
        retrieval_rows.append({"query":query,"rank":rank,"chunk_id":c["chunk_id"],"title":c["title"],
                               "cosine_similarity":float(score),"retrieved_text":c["text"]})
retrieval_df=pd.DataFrame(retrieval_rows)
display(retrieval_df)


QUERY: What technologies estimate body pose and joint angles?
Rank 1 | BC02-C1 | cosine=0.659 | Pose and biomechanics
The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its biomechanics module evaluates cues such as lead elbow elevation, lead knee flexion, head-over-knee balance, and spine forward lean.
Rank 2 | BC06-C1 | cosine=0.322 | Limitations and responsible use
Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence and the quality of the input video. A single recording cannot guarantee future match performance or diagnose a medical condition. Treat automated feedback as an aid for practice, not a replacement for a qualified coach.
Rank 3 | BC04-C1 | cosine=0.283 | Computer vision and shot classification
The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a VideoMAE transformer for classifying cricket shots across ten foundational shot types.

QUERY: How does the system track the b

,query,rank,chunk_id,title,cosine_similarity,retrieved_text
0,What technologies estimate body pose and joint...,1,BC02-C1,Pose and biomechanics,0.659236,The project describes MediaPipe-based pose est...
1,What technologies estimate body pose and joint...,2,BC06-C1,Limitations and responsible use,0.321618,Computer-vision feedback depends on camera ang...
2,What technologies estimate body pose and joint...,3,BC04-C1,Computer vision and shot classification,0.283450,The project describes YOLOv8-OBB for oriented ...
3,How does the system track the bat when a playe...,1,BC01-C1,Project overview,0.535487,BatCoach AI Pro is described as a browser-base...
4,How does the system track the bat when a playe...,2,BC04-C1,Computer vision and shot classification,0.517584,The project describes YOLOv8-OBB for oriented ...
5,How does the system track the bat when a playe...,3,BC03-C1,Practice modes,0.475838,The README describes shadow practice without a...
6,What happens when the same technique mistake i...,1,BC05-C1,Feedback and repeated errors,0.261472,The project describes on-screen coaching cues ...
7,What happens when the same technique mistake i...,2,BC06-C1,Limitations and responsible use,0.249859,Computer-vision feedback depends on camera ang...
8,What happens when the same technique mistake i...,3,BC03-C1,Practice modes,0.120093,The README describes shadow practice without a...
9,Can the webcam guarantee a century in the next...,1,BC06-C1,Limitations and responsible use,0.378362,Computer-vision feedback depends on camera ang...


In [ ]:
# Manually judge each returned chunk: Relevant / Partially Relevant / Irrelevant.
judgements=retrieval_df.copy()
judgements["Relevance judgement"]=""
judgements["Reason"]=""
display(judgements)
judgements.to_csv("experiment8_retrieval_judgements.csv",index=False)

,query,rank,chunk_id,title,cosine_similarity,retrieved_text,Relevance judgement,Reason
0,What technologies estimate body pose and joint...,1,BC02-C1,Pose and biomechanics,0.659236,The project describes MediaPipe-based pose est...,,
1,What technologies estimate body pose and joint...,2,BC06-C1,Limitations and responsible use,0.321618,Computer-vision feedback depends on camera ang...,,
2,What technologies estimate body pose and joint...,3,BC04-C1,Computer vision and shot classification,0.283450,The project describes YOLOv8-OBB for oriented ...,,
3,How does the system track the bat when a playe...,1,BC01-C1,Project overview,0.535487,BatCoach AI Pro is described as a browser-base...,,
4,How does the system track the bat when a playe...,2,BC04-C1,Computer vision and shot classification,0.517584,The project describes YOLOv8-OBB for oriented ...,,
5,How does the system track the bat when a playe...,3,BC03-C1,Practice modes,0.475838,The README describes shadow practice without a...,,
6,What happens when the same technique mistake i...,1,BC05-C1,Feedback and repeated errors,0.261472,The project describes on-screen coaching cues ...,,
7,What happens when the same technique mistake i...,2,BC06-C1,Limitations and responsible use,0.249859,Computer-vision feedback depends on camera ang...,,
8,What happens when the same technique mistake i...,3,BC03-C1,Practice modes,0.120093,The README describes shadow practice without a...,,
9,Can the webcam guarantee a century in the next...,1,BC06-C1,Limitations and responsible use,0.378362,Computer-vision feedback depends on camera ang...,,
